In [2]:
!pip install fvcore av pytorchvideo

/kaggle/input/datasets/nazymkuandykova/wlasl-300/WLASL_300

In [3]:
import os
import cv2
import time
import random
import math
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

DATA_DIR        = "/kaggle/input/datasets/nazymkuandykova/wlasl-300/WLASL_300"
BATCH_SIZE      = 8
LR              = 1e-4
WD              = 1e-4
EPOCHS          = 30
SAVE_PATH       = "slowfast_wlasl300_fine_tuned.pth"
device          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_FRAMES      = 32
TRAIN_SAMPLING_RATE  = 2   
TEST_SAMPLING_RATE   = 1
ALPHA           = 4
SLOW_FRAMES     = NUM_FRAMES // ALPHA   # 8
FAST_FRAMES     = NUM_FRAMES            # 32
TARGET_FPS      = 30
TRAIN_CROP_SIZE = 224
TEST_CROP_SIZE  = 256
TRAIN_MIN_SCALE = 256
TRAIN_MAX_SCALE = 320
MEAN            = [0.45, 0.45, 0.45]
STD             = [0.225, 0.225, 0.225]

TRAIN_CLIPS         = 1
VAL_CLIPS           = 1
TEST_TEMPORAL_CLIPS = 2   
TEST_SPATIAL_CROPS  = 1   
CROP_JITTER         = 20


def get_start_end_idx(video_size, clip_size, clip_idx, num_clips):
    delta = max(video_size - clip_size, 0)
    if clip_idx == -1:
        start_idx = random.uniform(0, delta)
    else:
        start_idx = delta * clip_idx / num_clips
    end_idx = start_idx + clip_size - 1
    return start_idx, end_idx


def temporal_sampling(frames, start_idx, end_idx, num_samples):
    indices = np.linspace(start_idx, end_idx, num_samples)
    indices = np.clip(indices, 0, len(frames) - 1).astype(int)
    return frames[indices]


def load_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    frames = []
    while True:
        ret, f = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    
    if len(frames) == 0:
        frames = [np.zeros((224, 224, 3), dtype=np.uint8)]
    
    return np.stack(frames), len(frames), fps

def sample_frames(video_path, num_frames, sampling_rate, clip_idx, num_clips):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or TARGET_FPS
    cap.release()
    
    clip_size = max(1, math.ceil(sampling_rate * (num_frames - 1) / TARGET_FPS * fps))
    
    frames, total_frames, _ = load_video_frames(video_path)
    
    start_idx, end_idx = get_start_end_idx(len(frames), clip_size, clip_idx, num_clips)
    
    return temporal_sampling(frames, start_idx, end_idx, num_frames)


def short_side_scale(frames, size):
    h, w  = frames.shape[1], frames.shape[2]
    short = min(h, w)
    scale = size / short
    new_h, new_w = int(h * scale), int(w * scale)
    return np.stack([cv2.resize(f, (new_w, new_h)) for f in frames])


def center_crop(frames, crop_size):
    h, w = frames.shape[1], frames.shape[2]
    top  = (h - crop_size) // 2
    left = (w - crop_size) // 2
    return frames[:, top:top + crop_size, left:left + crop_size]


def random_short_side_scale(frames, min_size, max_size):
    h, w      = frames.shape[1], frames.shape[2]
    new_short = np.random.randint(min_size, max_size + 1)
    if h < w:
        new_h, new_w = new_short, int(w * new_short / h)
    else:
        new_h, new_w = int(h * new_short / w), new_short
    return np.stack([cv2.resize(f, (new_w, new_h)) for f in frames])

def random_crop(frames, crop_size):
    h, w = frames.shape[1], frames.shape[2]
    top  = random.randint(0, h - crop_size)
    left = random.randint(0, w - crop_size)
    return frames[:, top:top + crop_size, left:left + crop_size]


def random_horizontal_flip(frames, prob=0.5):
    if random.random() < prob:
        return frames[:, :, ::-1, :].copy()  
    return frames
    
def spatial_sampling(frames, mode):
    if mode == 'train':
        frames = random_short_side_scale(frames, TRAIN_MIN_SCALE, TRAIN_MAX_SCALE)
        frames = random_crop(frames, TRAIN_CROP_SIZE)
        frames = random_horizontal_flip(frames, prob=0.5)
    elif mode == 'val':
        frames = random_short_side_scale(frames, TRAIN_MIN_SCALE, TRAIN_MAX_SCALE)
        frames = random_crop(frames, TRAIN_CROP_SIZE)
    else:  
        frames = short_side_scale(frames, TEST_CROP_SIZE)
        frames = center_crop(frames, TEST_CROP_SIZE)  
    return frames


def normalize(frames):
    x    = frames.astype(np.float32) / 255.0
    mean = np.array(MEAN, dtype=np.float32)
    std  = np.array(STD,  dtype=np.float32)
    return (x - mean) / std


def to_slow_fast(frames_fast):
    fast = torch.from_numpy(frames_fast).permute(3, 0, 1, 2).float()
    slow = torch.index_select(
        fast,
        1,
        torch.linspace(
            0, fast.shape[1] - 1, fast.shape[1] // ALPHA
        ).long()
    )
    return [slow, fast]


class WLASLDataset(Dataset):

    def __init__(self, root_dir, mode='train'):
        self.mode = mode
        split_dir = os.path.join(root_dir, mode)
        if not os.path.exists(split_dir):
            raise FileNotFoundError(f"Path not found: {split_dir}")

        self.classes      = sorted(os.listdir(split_dir))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        base_samples = []
        for cls in self.classes:
            cls_path = os.path.join(split_dir, cls)
            for vid in os.listdir(cls_path):
                if vid.endswith('.mp4'):
                    base_samples.append(
                        (os.path.join(cls_path, vid), self.class_to_idx[cls]))

        if mode == 'test':
            self.samples = [
                (path, label, t_idx, s_idx)
                for path, label in base_samples
                for t_idx in range(TEST_TEMPORAL_CLIPS)
                for s_idx in range(TEST_SPATIAL_CROPS)
            ]
        else:
            self.samples = [
                (path, label)
                for path, label in base_samples
            ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.mode == 'test':
            path, label, t_idx, s_idx = self.samples[idx]
            frames = sample_frames(
                path, FAST_FRAMES, TEST_SAMPLING_RATE,  
                clip_idx=t_idx, num_clips=TEST_TEMPORAL_CLIPS)
            frames = spatial_sampling(frames, mode='test')
        else:
            path, label = self.samples[idx]
            frames = sample_frames(
                path, FAST_FRAMES, TRAIN_SAMPLING_RATE,  
                clip_idx=-1, num_clips=1)
            frames = spatial_sampling(frames, mode=self.mode)
        
        frames = normalize(frames)
        slow, fast = to_slow_fast(frames)
        return [slow, fast], label


def build_slowfast_wlasl(num_classes=300):
    model = torch.hub.load('facebookresearch/pytorchvideo',
                           'slowfast_r50', pretrained=True)
    in_features = model.blocks[6].proj.in_features
    model.blocks[6].proj    = nn.Linear(in_features, num_classes)
    model.blocks[6].dropout = nn.Dropout(p=0.5)
    return model


train_loader = DataLoader(WLASLDataset(DATA_DIR, 'train'),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(WLASLDataset(DATA_DIR, 'val'),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(WLASLDataset(DATA_DIR, 'test'),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

model     = build_slowfast_wlasl(num_classes=300).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')

best_val_acc = 0.0
train_losses = []
val_losses = []
train_accuracies = []    
val_accuracies = []      
train_errors = []        
val_errors = []          
total_train_start = time.time()
epoch_durations = []

print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc%':<12} {'Train Err%':<12} "
      f"{'Val Loss':<12} {'Val Acc%':<12} {'Val Err%':<12} {'Time(s)':<10}")
print("-" * 95)

for epoch in range(EPOCHS):
    epoch_start = time.time()

    model.train()
    train_loss = train_correct = train_total = 0
    for inputs, labels in train_loader:
        inputs = [i.to(device) for i in inputs]
        labels = labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = [i.to(device) for i in inputs]
            labels = labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item()
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)

    train_acc = 100 * train_correct / train_total
    train_error = 100 - train_acc
    
    val_acc = 100 * val_correct / val_total
    val_error = 100 - val_acc

    train_losses.append(train_loss / len(train_loader))
    val_losses.append(val_loss / len(val_loader))
    scheduler.step()

    duration  = time.time() - epoch_start
    epoch_durations.append(duration)

    save_msg    = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        save_msg = "*"

    print(f"{epoch+1:<8} {train_loss/len(train_loader):<12.4f} "
          f"{train_acc:<12.2f} {train_error:<12.2f} "
          f"{val_loss/len(val_loader):<12.4f} {val_acc:<12.2f} {val_error:<12.2f} "
          f"{duration:<10.2f}")

total_time = time.time() - total_train_start
print("-" * 85)
print(f"Training Finished!")
print(f"Overall Training Time: {total_time / 60:.2f} minutes")
print(f"Average Time per Epoch: {sum(epoch_durations)/len(epoch_durations):.2f} seconds")
print("-" * 85)

print("Running Final Evaluation on TEST set...")
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

from collections import defaultdict
video_probs_sum = defaultdict(lambda: torch.zeros(300))
video_clip_count = defaultdict(int)
video_labels = {}
test_dataset = test_loader.dataset

with torch.no_grad():
    sample_idx = 0
    for inputs, labels in test_loader:
        inputs = [i.to(device) for i in inputs]
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1).cpu()
        for b in range(labels.size(0)):
            path, lbl, t_idx, s_idx = test_dataset.samples[sample_idx]
            video_probs_sum[(path, lbl)] += probs[b]
            video_clip_count[(path, lbl)] += 1
            video_labels[(path, lbl)] = lbl
            sample_idx += 1

correct_top1 = 0
correct_top5 = 0
for key, prob_sum in video_probs_sum.items():
    label = video_labels[key]
    avg_probs = prob_sum / video_clip_count[key]  # Explicit averaging
    correct_top1 += avg_probs.argmax().item() == label
    correct_top5 += label in avg_probs.topk(5).indices.tolist()

total = len(video_probs_sum)
test_top1_acc = 100 * correct_top1 / total
test_top1_error = 100 - test_top1_acc
test_top5_acc = 100 * correct_top5 / total
test_top5_error = 100 - test_top5_acc

print(f"\n{'='*50}")
print(f"FINAL TEST RESULTS:")
print(f"{'='*50}")
print(f"Top-1 Accuracy:  {test_top1_acc:.2f}%")
print(f"Top-1 Error:     {test_top1_error:.2f}%")
print(f"Top-5 Accuracy:  {test_top5_acc:.2f}%")
print(f"Top-5 Error:     {test_top5_error:.2f}%")
print(f"{'='*50}")

Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/SLOWFAST_8x8_R50.pyth" to /root/.cache/torch/hub/checkpoints/SLOWFAST_8x8_R50.pyth


100%|██████████| 264M/264M [00:00<00:00, 350MB/s] 


Epoch    Train Loss   Train Acc%   Train Err%   Val Loss     Val Acc%     Val Err%     Time(s)   
-----------------------------------------------------------------------------------------------
1        5.7045       0.87         99.13        5.5984       1.11         98.89        363.15    
2        5.3705       3.10         96.90        4.7783       8.21         91.79        363.83    
3        4.5188       10.90        89.10        3.9576       19.98        80.02        363.87    
4        3.7312       25.89        74.11        3.4656       34.52        65.48        363.11    
5        3.1718       39.05        60.95        3.0754       41.73        58.27        363.25    
6        2.7202       53.09        46.91        2.8440       51.05        48.95        363.40    
7        2.3489       64.44        35.56        2.7189       55.49        44.51        364.15    
8        2.1044       73.12        26.88        2.5639       58.16        41.84        362.85    
9        1.8556       